# tf-idf


In [5]:
corpus = ['我们都生活在阴沟里，但仍有人仰望星空。',
            '每个圣人都有过去，每个罪人都有未来。']

## 分词

In [6]:
import jieba

In [7]:
stopwords_filepath = 'data/stopwords.txt'
stopwords = [line.strip() for line in open(stopwords_filepath, 'r', encoding='utf-8').readlines()]

In [8]:
words_list = []
for doc in corpus:
    seg_list = jieba.cut(doc)
    words = [word for word in seg_list if word not in stopwords and len(word) >= 1]
    words_list.append(words)
print(words_list)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.305 seconds.
Prefix dict has been built successfully.


[['我们', '都', '生活', '在', '阴沟', '里', '但', '仍', '有人', '仰望', '星空'], ['每个', '圣人', '都', '有', '过去', '每个', '罪人', '都', '有', '未来']]


## 词袋模型

In [9]:
from gensim import corpora
dictionary = corpora.Dictionary(words_list)
print(dictionary.token2id)

{'仍': 0, '仰望': 1, '但': 2, '在': 3, '我们': 4, '星空': 5, '有人': 6, '生活': 7, '都': 8, '里': 9, '阴沟': 10, '圣人': 11, '有': 12, '未来': 13, '每个': 14, '罪人': 15, '过去': 16}


In [11]:
words_bag = [dictionary.doc2bow(word) for word in words_list]
print(words_bag)

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1)], [(8, 2), (11, 1), (12, 2), (13, 1), (14, 2), (15, 1), (16, 1)]]


(0,1) ----> “仍”出现一次
(8, 2) ----> “都”出现两次

(0,1)  ----> [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1)]

[1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0]

## tf-idf

In [12]:
from gensim import models

In [14]:
tfidf_model = models.TfidfModel(words_bag)
tfidf_model.save('data/tfidf_model.bin')

In [15]:
tfidf_model = models.TfidfModel.load('data/tfidf_model.bin')

In [18]:
tfidf_vec = [tfidf_model[doc] for doc in words_bag]
print(tfidf_vec)

[[(0, np.float64(0.31622776601683794)), (1, np.float64(0.31622776601683794)), (2, np.float64(0.31622776601683794)), (3, np.float64(0.31622776601683794)), (4, np.float64(0.31622776601683794)), (5, np.float64(0.31622776601683794)), (6, np.float64(0.31622776601683794)), (7, np.float64(0.31622776601683794)), (9, np.float64(0.31622776601683794)), (10, np.float64(0.31622776601683794))], [(11, np.float64(0.2886751345948129)), (12, np.float64(0.5773502691896258)), (13, np.float64(0.2886751345948129)), (14, np.float64(0.5773502691896258)), (15, np.float64(0.2886751345948129)), (16, np.float64(0.2886751345948129))]]


## 练习

##### 初始化

In [19]:
import jieba
from gensim import corpora, models

In [25]:
demo_txt = [
     "Apple iPhone 8 Plus （A1864） 64G8 深空灰色 移动联通电信4G手机",
     "Apple iPhone 7 Plus （AL66l） 128G 黑色 移动联通电信4G手机",
     "OPPO KTx 双模5G 4800万四摄 5000mAh长续航 90Hz电竞屏 蓝影 6GB+128GB 30W闪充 全阿通游戏智能手机",
     "一加手机 OnePlus ST 5G旗舰 120Hz 柔性直屏 65W闪充 高通税龙865 超强四摄 12GB+256GB 青域 拍照游戏手机",
     "小米 红米5 Plus 全面屏拍照手机 全网通版 3GB+32CB 金色 移动联通化信4G手机 双卡双待",
     "Apple iPhone 7 （A1660） 128G 黑色 移动联通电信4G手机",
     "Apple iPhone X （A1865） 64GB 深空灰色 移动联通电信4G手机",
     "小米 红米Note5A 移动4G+版全网通 4GB+64GB 铂银灰 移动联通电信4G手机 双卡双待 拍照手机",
     "荣耀 V10全网通 标配版 4GB+64GB 幻夜黑 移动联通电信4G全画屏手机 双卡双待",
     "Redmi Note 9 5G 天玑800U 18W快充 4800万超清三摄 云墨灰 6GB+128GB 游戏智能手机 小米 红米",
     "Apple iPhone 6s Plus （A1699） 128G 玫瑰金色 移动联通电信4G手机",
     "Apple iPhone 6 32GB 金色 移动联通电信4G手机",
     "小米Note3 美颜双摄拍照手机 6GB+64GB 黑色 全网通4G手机 双卡双待",
     "Apple iPhone 12（A2404） 128GB 黑色支持移动联通电信5G 双卡双待手机",
     "荣耀10 GT游戏加速 AIS手持夜聚 6GB+64GB 幻影蓝全网通 移动联通电信4G 双卡双待"
]

#### 分词

##### 自定义分词词典

手机文本中包含大量品牌名、型号、颜色、技术术语等，jieba 默认词典无法正确识别，需要添加自定义词典。


In [28]:
# 加载自定义词典
jieba.load_userdict('data/phone_dict.txt')

# 分词
stopwords_filepath = 'data/stopwords.txt'
stopwords = [line.strip() for line in open(stopwords_filepath, 'r', encoding='utf-8').readlines()]

words_list = []
for doc in demo_txt:
    seg_list = jieba.cut(doc)
    words = [word for word in seg_list if word not in stopwords and word != ' ']
    words_list.append(words)
    
# 打印前3条结果看看效果
for i, words in enumerate(words_list[:3]):
    print(f"文本{i+1}: {' | '.join(words)}")

文本1: Apple | iPhone | Plus | A1864 | 64G8 | 深空灰 | 色 | 移动 | 联通 | 电信 | 4G | 手机
文本2: Apple | iPhone | Plus | AL66l | 128G | 黑色 | 移动 | 联通 | 电信 | 4G | 手机
文本3: OPPO | KTx | 双模 | 5G | 4800万 | 四摄 | 5000mAh | 长 | 续航 | 90Hz | 电竞屏 | 蓝影 | 6GB | 128GB | 30W | 闪充 | 全阿通 | 游戏 | 智能手机


In [29]:
# 打印全部分词结果
for i, words in enumerate(words_list):
    print(f"文本{i+1}: {' | '.join(words)}")

文本1: Apple | iPhone | Plus | A1864 | 64G8 | 深空灰 | 色 | 移动 | 联通 | 电信 | 4G | 手机
文本2: Apple | iPhone | Plus | AL66l | 128G | 黑色 | 移动 | 联通 | 电信 | 4G | 手机
文本3: OPPO | KTx | 双模 | 5G | 4800万 | 四摄 | 5000mAh | 长 | 续航 | 90Hz | 电竞屏 | 蓝影 | 6GB | 128GB | 30W | 闪充 | 全阿通 | 游戏 | 智能手机
文本4: 一加 | 手机 | OnePlus | ST | 5G | 旗舰 | 120Hz | 柔性 | 直屏 | 65W | 闪充 | 高通 | 税龙 | 865 | 超强 | 四摄 | 12GB | 256GB | 青域 | 拍照 | 游戏手机
文本5: 小米 | 红米 | Plus | 全面屏 | 拍照手机 | 全网通 | 版 | 3GB | 32CB | 金色 | 移动 | 联通 | 化信 | 4G | 手机 | 双卡双待
文本6: Apple | iPhone | A1660 | 128G | 黑色 | 移动 | 联通 | 电信 | 4G | 手机
文本7: Apple | iPhone | X | A1865 | 64GB | 深空灰 | 色 | 移动 | 联通 | 电信 | 4G | 手机
文本8: 小米 | 红米Note5A | 移动 | 4G | 版 | 全网通 | 4GB | 64GB | 铂银灰 | 移动 | 联通 | 电信 | 4G | 手机 | 双卡双待 | 拍照手机
文本9: 荣耀 | V10 | 全网通 | 标配 | 版 | 4GB | 64GB | 幻夜黑 | 移动 | 联通 | 电信 | 4G | 全 | 画屏 | 手机 | 双卡双待
文本10: Redmi | Note | 5G | 天玑800U | 18W | 快充 | 4800万 | 超清三摄 | 云墨灰 | 6GB | 128GB | 游戏 | 智能手机 | 小米 | 红米
文本11: Apple | iPhone | 6s | Plus | A1699 | 128G | 玫瑰金 | 色 | 移动 | 联通 | 电信 | 4G | 手机
文本12:

#### 词袋模型

In [31]:
dictionary = corpora.Dictionary(words_list)
print(dictionary.token2id)

{'4G': 0, '64G8': 1, 'A1864': 2, 'Apple': 3, 'Plus': 4, 'iPhone': 5, '手机': 6, '深空灰': 7, '电信': 8, '移动': 9, '联通': 10, '色': 11, '128G': 12, 'AL66l': 13, '黑色': 14, '128GB': 15, '30W': 16, '4800万': 17, '5000mAh': 18, '5G': 19, '6GB': 20, '90Hz': 21, 'KTx': 22, 'OPPO': 23, '全阿通': 24, '双模': 25, '四摄': 26, '智能手机': 27, '游戏': 28, '电竞屏': 29, '续航': 30, '蓝影': 31, '长': 32, '闪充': 33, '120Hz': 34, '12GB': 35, '256GB': 36, '65W': 37, '865': 38, 'OnePlus': 39, 'ST': 40, '一加': 41, '拍照': 42, '旗舰': 43, '柔性': 44, '游戏手机': 45, '直屏': 46, '税龙': 47, '超强': 48, '青域': 49, '高通': 50, '32CB': 51, '3GB': 52, '全网通': 53, '全面屏': 54, '化信': 55, '双卡双待': 56, '小米': 57, '拍照手机': 58, '版': 59, '红米': 60, '金色': 61, 'A1660': 62, '64GB': 63, 'A1865': 64, 'X': 65, '4GB': 66, '红米Note5A': 67, '铂银灰': 68, 'V10': 69, '全': 70, '幻夜黑': 71, '标配': 72, '画屏': 73, '荣耀': 74, '18W': 75, 'Note': 76, 'Redmi': 77, '云墨灰': 78, '天玑800U': 79, '快充': 80, '超清三摄': 81, '6s': 82, 'A1699': 83, '玫瑰金': 84, '32GB': 85, 'Note3': 86, '美颜双摄': 87, '12': 88, 'A2404': 89, '

In [33]:
word_bag = [dictionary.doc2bow(word) for word in words_list]
print(word_bag)

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)], [(0, 1), (3, 1), (4, 1), (5, 1), (6, 1), (8, 1), (9, 1), (10, 1), (12, 1), (13, 1), (14, 1)], [(15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1)], [(6, 1), (19, 1), (26, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 1), (49, 1), (50, 1)], [(0, 1), (4, 1), (6, 1), (9, 1), (10, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 1), (56, 1), (57, 1), (58, 1), (59, 1), (60, 1), (61, 1)], [(0, 1), (3, 1), (5, 1), (6, 1), (8, 1), (9, 1), (10, 1), (12, 1), (14, 1), (62, 1)], [(0, 1), (3, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (63, 1), (64, 1), (65, 1)], [(0, 2), (6, 1), (8, 1), (9, 2), (10, 1), (53, 1), (56, 1), (57, 1), (58, 1), (59, 1), (63, 1), (66, 1), (67, 1

#### tfidf

In [35]:
tfidf_model = models.TfidfModel(words_bag)
tfsave_path = 'data/tfidf_model_phone.bin'
tfidf_model.save(tfsave_path)

In [39]:
tfidf_vectors = [tfidf_model[word] for word in word_bag]
print(tfidf_vectors[0])

[(0, np.float64(0.30151134457776363)), (1, np.float64(0.30151134457776363)), (2, np.float64(0.30151134457776363)), (3, np.float64(0.30151134457776363)), (4, np.float64(0.30151134457776363)), (5, np.float64(0.30151134457776363)), (6, np.float64(0.30151134457776363)), (7, np.float64(0.30151134457776363)), (9, np.float64(0.30151134457776363)), (10, np.float64(0.30151134457776363)), (11, np.float64(0.30151134457776363))]
